# 生成经验回放数据（防止灾难性遗忘）

用 Qwen3-4B 自身生成 60 条通用问答数据，微调时混入训练集，让模型不会忘记基础能力。

**原理：** 微调时只训练角色扮演数据会导致模型遗忘通用知识。混入一批由模型自己生成的通用问答，相当于告诉模型「这些回答我以前就会，继续保持」。

In [ ]:
! pip install -q -U transformers bitsandbytes accelerate

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"GPU可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 加载模型

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    print("pad_token 为空，设置为 eos_token")
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("模型加载完成")

# 定义提示词

12类 × 5条 = 60条

In [ ]:
# 12个话题类别，每类5个提示，总共60条
# 涵盖知识、推理、写作等核心能力，防止微调后这些能力退化

PROMPTS = [
    # --- 1. 日常知识问答 ---
    "为什么天空是蓝色的？",
    "冰箱里食物一般能保存多久？",
    "为什么打哈欠会传染？",
    "人一天需要喝多少水比较合适？",
    "为什么飞机窗户是椭圆形的？",

    # --- 2. 数学与逻辑推理 ---
    "一个正方形的面积是64平方厘米，它的周长是多少？",
    "如果今天是周三，30天后是星期几？",
    "0.1 + 0.2 等于多少？用Python计算会是什么结果，为什么？",
    "有100个台阶，每次可以走1步或2步，共有多少种走法？",
    "一个房间里有3盏灯，室外有3个开关，你只能进房间一次，如何判断哪个开关控制哪盏灯？",

    # --- 3. 中文语言与写作 ---
    "帮我把这句话改得更正式一些：'这个方案不太行，换个思路吧'",
    "解释一下'塞翁失马，焉知非福'这句话的意思",
    "用5个词语描述一个春天的早晨",
    "'的地得'的用法有什么区别？举例说明",
    "写一个50字以内的自我介绍，适合在新公司入职时使用",

    # --- 4. 历史与文化 ---
    "简单介绍一下丝绸之路的历史意义",
    "中国的四大发明是什么？各有什么影响？",
    "唐朝和宋朝在经济上有什么主要区别？",
    "为什么春节要贴春联、放鞭炮？",
    "简述一下第二次世界大战结束的标志性事件",

    # --- 5. 科普常识 ---
    "地球上最深的海沟是哪里？有多深？",
    "光速是多少？光从太阳到地球需要多长时间？",
    "为什么人会做梦？",
    "DNA和RNA有什么区别？",
    "黑洞是什么？简单解释一下",

    # --- 6. 生活技巧 ---
    "衣服上沾了红酒怎么快速处理？",
    "如何快速让一杯水变凉但不稀释它？",
    "手机充电有什么保护电池的小技巧？",
    "如何快速记住一个新认识的人的名字？",
    "眼睛疲劳时有哪些简单的缓解方法？",

    # --- 7. 职场与学习 ---
    "如何更高效地做会议记录？",
    "番茄工作法是什么？怎么用？",
    "和难相处的同事合作时，有什么沟通技巧？",
    "如何在30分钟内快速理解一篇技术文章？",
    "面试时被问到'你最大的缺点是什么'，怎么回答比较好？",

    # --- 8. 健康与饮食 ---
    "每天坐着工作8小时，有哪些简单的运动可以在办公室做？",
    "早餐吃什么比较健康又方便？",
    "睡眠质量不好，有哪些改善方法？",
    "感冒初期应该怎么处理？",
    "减少糖分摄入有什么实际的好处？",

    # --- 9. 编程与技术 ---
    "Python中列表和元组有什么区别？什么时候用哪个？",
    "什么是REST API？举个例子说明",
    "Git中merge和rebase有什么区别？",
    "解释一下什么是递归，用一个简单例子说明",
    "SQL中LEFT JOIN和INNER JOIN有什么区别？",

    # --- 10. 逻辑谜题 ---
    "鸡兔同笼：头共10个，脚共28只，鸡和兔各有几只？",
    "一个人花30元买了一只鸡，40元卖出，50元买回，60元卖出，请问他赚了多少钱？",
    "有一根蜡烛，从两头同时点，1小时烧完，只点一头需要多长时间？",
    "如果5台机器5分钟生产5个零件，100台机器100分钟能生产多少个零件？",
    "一个池子，单开进水管需要6小时注满，单开出水管需要4小时放完，同时开需要多长时间放完？",

    # --- 11. 情感与人际 ---
    "朋友最近情绪低落，我该怎么安慰他/她？",
    "如何礼貌地拒绝别人的请求而不伤害感情？",
    "和家人意见不合时，如何有效沟通？",
    "如何在新环境中快速融入一个团队？",
    "长期异地恋有哪些维持感情的方法？",

    # --- 12. 创意与头脑风暴 ---
    "如果你要开一家独特的咖啡店，有什么创意主题推荐？",
    "给我三个既实用又有创意的生日礼物点子，预算200元以内",
    "假如你要写一部短篇科幻小说，你会选什么主题？简单描述一下故事梗概",
    "如何把日常通勤的无聊时间变得更有价值？",
    "你能想到哪些用手机可以做的有意思的小项目？",
]

print(f"共 {len(PROMPTS)} 个提示词，涵盖 12 个类别")

# 批量生成回答

In [ ]:
def generate_response(prompt: str, max_new_tokens: int = 512) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    return response.strip()


# 生成所有回答
replay_data = []
failed = []

print(f"开始生成，共 {len(PROMPTS)} 条...\n")

for i, prompt in enumerate(PROMPTS):
    try:
        response = generate_response(prompt)
        # 过滤空回答
        if len(response.strip()) < 5:
            raise ValueError(f"回答过短: {repr(response)}")

        replay_data.append({
            "conversations": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": response},
            ]
        })

        # 每10条打印一次进度
        if (i + 1) % 10 == 0 or i == 0:
            print(f"[{i+1}/{len(PROMPTS)}] Q: {prompt[:30]}...")
            print(f"         A: {response[:60]}...")
            print()

    except Exception as e:
        print(f"[{i+1}] 失败: {prompt[:30]}... 错误: {e}")
        failed.append(i)

print(f"\n生成完成：成功 {len(replay_data)} 条，失败 {len(failed)} 条")

# 验证质量

In [ ]:
# 抽查几条，人工确认质量
import random

samples = random.sample(replay_data, min(5, len(replay_data)))
print("=== 随机抽查 5 条 ===")
for s in samples:
    q = s["conversations"][0]["content"]
    a = s["conversations"][1]["content"]
    print(f"Q: {q}")
    print(f"A: {a[:200]}{'...' if len(a) > 200 else ''}")
    print("-" * 60)

# 保存数据

In [ ]:
OUTPUT_FILE = "replay_data.json"

# 保存为和 train_data_augmented.json 兼容的格式
# system_prompt 留空，微调时不注入角色 prompt，让模型保持通用能力
output = {
    "system_prompt": "",
    "conversations": replay_data,
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"已保存 {len(replay_data)} 条数据到 {OUTPUT_FILE}")
print(f"文件大小: {__import__('os').path.getsize(OUTPUT_FILE) / 1024:.1f} KB")

In [ ]:
# 下载到本地
from google.colab import files
files.download(OUTPUT_FILE)
print(f"已下载 {OUTPUT_FILE}")